In [1]:
# !pip install ultrack -q
# !pip install tifffile -q

In [ ]:
import tifffile
import numpy as np
from ultrack import MainConfig, load_config, Tracker, track, to_tracks_layer, tracks_to_zarr
from ultrack.imgproc import robust_invert, detect_foreground
from ultrack.utils.array import array_apply, create_zarr
from ultrack.utils.cuda import import_module, to_cpu, on_gpu, torch_default_device, is_cupy_array
import zarr

detect_foreground = on_gpu(detect_foreground)
# robust_invert = on_gpu(robust_invert)
track = on_gpu(track)

In [3]:
torch_default_device()

device(type='cuda', index=0)

In [4]:
import os

# moce into the directory containing the data
os.chdir("/home/jschauser/cellTracking/")

In [5]:


def check_if_zarr_exists(path: str) -> bool:
    try:
        zarr.open(path, mode="r")
        return True
    except Exception:
        return False



In [6]:


# load in image
p = "data/MATTEO to ALA/Embryo_37_intrareg_fuse_t0"
paths = [str(i) for i in range(75,98)]  # list of timepoints


subsample = 2

print("Loading images...")
imgs = []
for path in paths:
  print(f"Loading {path}...")
  image = tifffile.imread(p + path + ".tif")
  imgs.append(image[::subsample,::subsample, ::subsample])


all_images = np.array(imgs)
print("")

voxel_size = None#[1,1,1]

images = all_images



Loading images...
Loading 75...
Loading 76...
Loading 77...
Loading 78...
Loading 79...
Loading 80...
Loading 81...
Loading 82...
Loading 83...
Loading 84...
Loading 85...
Loading 86...
Loading 87...
Loading 88...
Loading 89...
Loading 90...
Loading 91...
Loading 92...
Loading 93...
Loading 94...
Loading 95...
Loading 96...
Loading 97...



In [7]:

for img in images:
    print("Image shape:", img.shape)
    break

Image shape: (395, 1004, 423)


In [8]:

fg0 = detect_foreground(images[0])


print(fg0.shape, fg0.dtype)
print(fg0.nbytes / 1e9, "GB per image")


(395, 1004, 423) bool
0.16775334 GB per image


In [9]:
import zarr

foregrounds = zarr.open(
    "foregrounds.zarr",
    mode="w",
    shape=images.shape,
    dtype=bool,
    chunks=(1, 100, 512, 512)
)

for i, img in enumerate(images):
    print(f"Processing image {i+1}/{len(images)}...")
    foregrounds[i] = detect_foreground(img)


Processing image 1/23...
Processing image 2/23...
Processing image 3/23...
Processing image 4/23...
Processing image 5/23...
Processing image 6/23...
Processing image 7/23...
Processing image 8/23...
Processing image 9/23...
Processing image 10/23...
Processing image 11/23...
Processing image 12/23...
Processing image 13/23...
Processing image 14/23...
Processing image 15/23...
Processing image 16/23...
Processing image 17/23...
Processing image 18/23...
Processing image 19/23...
Processing image 20/23...
Processing image 21/23...
Processing image 22/23...
Processing image 23/23...


In [19]:
boundaries = zarr.open(
    "boundaries.zarr",
    mode="w",
    shape=foregrounds.shape,
    dtype=np.float16,
    chunks=(1, 100, 512, 512),
)


for i in range(foregrounds.shape[0]):
    print(f"Processing image {i+1}/{foregrounds.shape[0]}...")
    fg = np.asarray(foregrounds[i]).astype(np.float16)  # force load to RAM
    boundaries[i] = robust_invert(fg)



Processing image 1/23...
Processing image 2/23...
Processing image 3/23...
Processing image 4/23...
Processing image 5/23...
Processing image 6/23...
Processing image 7/23...
Processing image 8/23...
Processing image 9/23...
Processing image 10/23...
Processing image 11/23...
Processing image 12/23...
Processing image 13/23...
Processing image 14/23...
Processing image 15/23...
Processing image 16/23...
Processing image 17/23...
Processing image 18/23...
Processing image 19/23...
Processing image 20/23...
Processing image 21/23...
Processing image 22/23...
Processing image 23/23...


In [ ]:

cfg =  MainConfig()  # or load default config
cfg.segmentation_config.threshold = 0.5
cfg.linking_config.max_distance = 15.0
cfg.linking_config.n_workers = 8
cfg.linking_config.max_neighbors = 10

# set solver to GUROBI
cfg.tracking_config.solver_name = "GUROBI"

print("Tracking...")

track(
    cfg,
    foreground=foregrounds,
    contours=boundaries,
    scale=voxel_size,
    overwrite="links",
)


Tracking...


Linking nodes.: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 22/22 [00:15<00:00,  1.39it/s]


Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2763812


Using Coin-OR Branch and Cut solver
Solving ILP batch 0
Constructing ILP ...
Solving ILP ...
Welcome to the CBC MILP Solver 
Version: Trunk
Build Date: Oct 24 2021 

Starting solution of the Linear programming relaxation problem using Primal Simplex

Coin0506I Presolve 4299273 (-66773) rows, 3379670 (-23432) columns and 13618472 (-144211) elements
Clp0030I 2 infeas 8.7382545, obj 25.862312 - mu 0.001, its 105, 953910 interior
Clp0030I 3 infeas 1.0751801, obj 53.279904 - mu 0.001, its 105, 1039238 interior
Clp0030I 4 infeas 4.5754058, obj 62.309227 - mu 0.0003333, its 105, 1116021 interior
Clp0030I 5 infeas 0.69504037, obj 72.509799 - mu 0.0003333, its 105, 1154685 interior
Clp0030I 6 infeas 0.8393169, obj 86.726573 - mu 0.0003333, its 105, 1173482 interior
Clp0030I 7 infeas 1.5214449, obj 90.610214 - mu 0.00011108889, its 105, 1198281 interior
Clp0030I 8 infeas 0.25014193, obj 94.847554 - mu 0.00011108889, its 105, 1202552 interior
Clp0030I 9 infeas 0.38165175, obj 99.823661 - mu 0.000

In [ ]:
from ultrack.core.linking.processing import link
from ultrack.core.linking.utils import clear_linking_data
from ultrack.core.segmentation.processing import segment
from ultrack.core.solve.processing import solve


segment(foregrounds, edges=boundaries)
link(cfg)
solve(cfg)


In [ ]:
# export to good format 
tracks_df, graph = to_tracks_layer(cfg)

# tracks_df = to_cpu(tracks_df)

tracks_df.to_csv("tracks.csv", index=False)

